# TP-MCTS Experiments

Two experiment scripts, run from this notebook.

| Script | Purpose |
|--------|---------|
| `run_mcts_heuristic_comparison.py` | TP-MCTS score across 4 NASA scenarios × 6 heuristics |
| `run_heuristic_runtime_per_call.py` | Per-call timing (wrapper + worker + cache hit/miss) |

**Setup:** clone the repo and `cd` into it first (same as `demo.ipynb` cells 1-4).

In [12]:
# Colab: fresh clone every time this cell runs (avoids stale /content/tp_mcts)
# On non-Colab machines, `/content` won't exist — this cell is a no-op there.
import os
import shutil
import subprocess
import sys

REPO_DIR = "/content/tp_mcts"
REPO_URL = "https://github.com/eliezerRevach/tp_mcts.git"
# After pushing fixed-tail (and related) commits, set this to your branch name, e.g. "claude/festive-poitras-5b02ce"
REPO_BRANCH = None  # None = default branch on GitHub

if not os.path.isdir("/content"):
    print("Skip Colab clone setup: not running on Colab (`/content` missing). cwd=", os.getcwd())
else:
    if not REPO_DIR.startswith("/content/"):
        raise RuntimeError(f"Refusing to delete unexpected path: {REPO_DIR}")

    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR, ignore_errors=True)

    os.chdir("/content")
    clone_cmd = ["git", "clone", REPO_URL, REPO_DIR]
    if REPO_BRANCH:
        clone_cmd = ["git", "clone", "--branch", str(REPO_BRANCH), REPO_URL, REPO_DIR]
    subprocess.check_call(clone_cmd)

    os.chdir(REPO_DIR)
    rev = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
    print("Git HEAD:", rev, ("branch=" + str(REPO_BRANCH)) if REPO_BRANCH else "")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "-q", "install", "dill", "numpy", "pandas", "openpyxl"]
    )
    print("Ready:", os.getcwd())

Ready: /content/tp_mcts


## Config

Edit the values below, then run the cells for each experiment.

In [15]:
# ── Shared config ─────────────────────────────────────────────────────────
from pathlib import Path
import itertools

def _is_tp_mcts_root(path: Path) -> bool:
    return (
        (path / "scripts" / "run_mcts_heuristic_comparison.py").is_file()
        and (path / "unified_planning" / "run_domain.py").is_file()
        and (path / "experiments.ipynb").is_file()
    )

def _find_repo_root() -> Path:
    """Locate the outermost TP-MCTS repo root, avoiding accidental nested clones."""
    # Colab: pin the standard clone path when present so cwd=/content (or elsewhere)
    # still resolves to the repo that `git clone` created — not a random cwd.
    colab_root = Path("/content/tp_mcts")
    if colab_root.is_dir() and _is_tp_mcts_root(colab_root):
        return colab_root.resolve()

    start = Path.cwd().resolve()
    candidates = [p for p in [start, *start.parents] if _is_tp_mcts_root(p)]
    if candidates:
        # If cwd is inside TP_MCTS/tp_mcts, both roots match; use the outer one.
        return candidates[-1]
    return start

REPO_ROOT = _find_repo_root()
NESTED_REPO = REPO_ROOT / "tp_mcts"
if (NESTED_REPO / ".git").exists():
    print(f"[warn] Nested repo copy exists and will be ignored: {NESTED_REPO}")

import sys
_scripts_dir = REPO_ROOT / "scripts"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(_scripts_dir) not in sys.path:
    sys.path.insert(0, str(_scripts_dir))
import importlib
import experiment_common as _experiment_common
importlib.reload(_experiment_common)
from experiment_common import HEURISTIC_ALIASES, validate_heuristics

def _require_colab_repo_has_fixed_tail():
    missing = [h for h in HEURISTICS if h.startswith("fixed_tail") and h not in HEURISTIC_ALIASES]
    if not missing:
        return
    raise RuntimeError(
        "Colab repo is missing fixed-tail aliases/heuristics: "
        + ", ".join(missing)
        + ". Push your latest tp_mcts to GitHub, set REPO_BRANCH in the setup cell, "
        "re-run setup + config (fresh clone). Local path: "
        + str(_experiment_common.__file__)
    )

RUNS                 = 10    # runs per (scenario × heuristic)
SEED                 = 123    # random seed — same seed across all runs
SEARCH_TIME          = 1      # MCTS search time per step in seconds; default in run_domain.py/script: 1
SEARCH_DEPTH         = 10   # MCTS tree depth; default in run_domain.py/script: 40; 0 = one-step/root-action test
K_RANDOM_ACTIONS     = 10   # --k for selection_type="max"; default in run_domain.py/script: 10; large value evaluates all actions
EXPLORATION_CONSTANT = 10 # UCT exploration constant C; default in run_domain.py: 10.0
SELECTION_TYPE       = "avg"  # default: "avg"; options: "avg", "rootInterval", "max"; use "max" with large K for root-greedy tests
REWARD_MODE          = "terminal"  # default: "deadline"; options: "deadline", "terminal"
VALUE_MODE           = "tp_mcts"  # default: "tp_mcts"; options: "tp_mcts", "greedy_matched", "ptrpg_guided_terminal_rollout", "fixed_tail_ptrpg_rollout"
FINAL_SELECTION      = "q"   # final action selection after MCTS search: "q"=argmax Q-value (default), "robust"=most-visited (argmax N)
SCRIPT3_SOLVER       = "greedy_parallel"  # Script 3 default; can be switched to "mcts" if needed
DOMAINS              = ["nasa_rover"]  # each must match `domains` in unified_planning/run_domain.py
DOMAIN               = DOMAINS[0]       # Script 1b uses one scenario, keep an alias for convenience
DISCOUNT_FACTOR      = 1     # MDP gamma; default in run_domain.py/script: 0.95; also --gamma on CLI
STEP_PENALTY         = 0     # reward added each transition; default in run_domain.py/script: -0.05
FIXED_TAIL_H         = SEARCH_DEPTH    # fixed_tail_* / value_mode=fixed_tail_ptrpg_rollout: common PTRPG tail horizon H
FIXED_TAIL_DEBUG     = False  # True => log first 5 fixed-tail leaf evaluations per MCTS search

# Domain argument capabilities used by Script 3 grid builder
DOMAINS_USING_OBJECTS = {"nasa_rover", "machine_shop", "hosting", "prob_match_cellar", "stuck_car"}
DOMAINS_USING_GARBAGE = {"prob_conc", "simple", "hosting"}

# Scenario grid — Script 1 only
OBJECTS         = [2]      # object_amount values
GARBAGE_AMOUNTS = [10]    # garbage_amount values (Script 3)
DEADLINES       = [25]       # deadline values

# Heuristics — used by both scripts (keys from scripts/experiment_common.py HEURISTIC_ALIASES)
# Full list of selectable heuristic keys:
#   basic / correlation:
#       ptrpg_old  baseline  baseline_cached  baseline_pessimistic  baseline_passmistic (typo alias -> pessimistic)
#   atom backtrack family:
#       atomic_exact
#       atomic_exact_resolution            (synonym: atom_backtrack_exact_resolution)            # exponential/log-spaced layers
#       atomic_exact_unbiased              (synonym: atom_backtrack_exact_unbiased)               # structural per-layer bias B(t)
#       atomic_exact_resolution_and_gamma  (synonym: atom_backtrack_exact_resolution_and_gamma)   # resolution + AND-layer gamma
#       atomic_exact_cached  fast_atom_cache
#   survival family (delete/survival decay so P_t can drop):
#       baseline_survival  baseline_survival_meanvar
#       baseline_survival_and_gamma        # survival + component-wise AND-layer gamma correction
#       baseline_survival_resolution       # survival over log-spaced (exponential-width) layers  P_{t-k}
#   alignment heuristics (fix cross-depth/deadline bias; use the *Alignment params* below):
#       rollout_aligned_baseline   rollout_aligned_survival   rollout_aligned_resolution_survival
#           # per-node value aligned to a common horizon via real prefix rollouts (dynamic parent-local H_p)
#       frontier_aligned_baseline  frontier_aligned_survival  frontier_aligned_resolution_survival
#       frontier_aligned_option_a  frontier_aligned_option_a_survival  frontier_aligned_option_a_resolution
#           # fresh global Option A; argmax aligned_value; no lambda; ignores fixed ROLLOUT_ALIGNED_H
#           # Option A: aligned value used as a frontier SELECTION score (blended with Q via lambda_align)
#   ptrpg_guided terminal rollout (MCTS backs up real 0/1 rollout; PTRPG only picks actions):
#       ptrpg_guided_rollout_baseline_survival_resolution
#       ptrpg_guided_rollout_atomic_exact_resolution
#           # aliases set value_mode=ptrpg_guided_terminal_rollout per row (keep VALUE_MODE=tp_mcts for baselines)
#           # use REWARD_MODE=terminal; disable ENABLE_ROLLOUT_CALIBRATION and alignment heuristics
#   fixed_tail PTRPG rollout (prefix to FIXED_TAIL_H, backup PTRPG(state, H); alias sets value_mode):
#       fixed_tail_atomic_exact_resolution
# First ptrpg-guided benchmark (copy/paste):
# HEURISTICS = [
#     "baseline",
#     "baseline_survival_resolution",
#     "atomic_exact_resolution",
#     "ptrpg_guided_rollout_baseline_survival_resolution",
#     "ptrpg_guided_rollout_atomic_exact_resolution",
# ]
# HEURISTICS   = ["ptrpg_old", "baseline", "baseline_cached", "baseline_pessimistic", "atomic_exact", "atomic_exact_resolution", "atomic_exact_unbiased", "atomic_exact_cached", "fast_atom_cache", "baseline_survival"]
HEURISTICS   = ["fixed_tail_atomic_exact_resolution"]
# ── New-heuristic params ──────────────────────────────────────────────────
# These ONLY affect the new heuristics; every shared MDP/MCTS param above
# (gamma=DISCOUNT_FACTOR, EXPLORATION_CONSTANT, SEARCH_TIME/DEPTH, K, reward/value
# mode, SEED, ...) stays identical across all heuristics so comparisons are fair.
# All defaults below leave the existing heuristics' behavior unchanged.

# baseline_survival_and_gamma: lazy rollout calibration of the AND-layer gamma
# factors. False = static gamma table (deterministic, fast). True = refine gamma
# from valid reachable rollouts during search (slower).
ENABLE_ROLLOUT_CALIBRATION = False  # True only for baseline_survival_and_gamma / *_and_gamma heuristics

# rollout_aligned_* and frontier_aligned_* (the alignment heuristics).
# Ignored by every non-aligned heuristic.
ROLLOUT_ALIGNED_FIXED_H  = False   # False = DYNAMIC horizon (the main mode): rollout_aligned_* uses the
                                   #         parent-local H_p, frontier_aligned_* uses H_frontier (deadline - deepest elapsed).
                                   # True  = use the fixed ROLLOUT_ALIGNED_H below (rollout_aligned_* ONLY; frontier ignores it).
ROLLOUT_ALIGNED_H        = 15      # FIXED suffix horizon. Used ONLY when ROLLOUT_ALIGNED_FIXED_H=True.
                                   # Has NO effect on the dynamic rollout_aligned_* or on frontier_aligned_*. (sweep 5/10/15/20)
ROLLOUT_ALIGNED_REDO     = 1       # amount of prefix rollouts averaged per node (sweep 1/5/10/20)
ROLLOUT_ALIGNED_BOUNDARY = "wait_no_overshoot"  # overshoot | wait_no_overshoot | expected_stochastic_rounding
# frontier_aligned_{baseline,survival,resolution}_* ONLY (lambda blend; ignored by frontier_aligned_option_a_*):
ROLLOUT_ALIGNED_LAMBDA_ALIGN = 1.0  # selection blend (1=aligned value, 0=plain UCT Q); sweep 0.25/0.5/0.75/1.0
FRONTIER_OPTION_A_DEBUG = False  # True => --frontier-option-a-debug (first 3 global selections)

# ptrpg_guided_terminal_rollout (value_mode or ptrpg_guided_rollout_* aliases):
PTRPG_GUIDED_ROLLOUT_MAX_STEPS = None  # None = use problem deadline (run_domain default)
PTRPG_GUIDED_ROLLOUT_EPSILON     = 0.0   # 0 = deterministic greedy rollout policy
PTRPG_GUIDED_ROLLOUT_DEBUG       = False  # True => log first rollout per MCTS search

# Optional safety budgets for prefix rollouts (0 = unlimited):
ROLLOUT_ALIGNED_MAX_ROLLOUTS_PER_NODE   = 0
ROLLOUT_ALIGNED_MAX_ROLLOUTS_PER_SEARCH = 0
ROLLOUT_ALIGNED_MAX_TIME_PER_SEARCH     = 0.0

# Resolution schedule (atom_backtrack_exact_resolution / atomic_exact_resolution /
# any *_resolution* heuristic):
#   user-facing "alpha"     → RESOLUTION_ALPHA          → forwarded as --resolution-alpha (None = omit = parser default)
#   user-facing "minimum"   → RESOLUTION_FORCED_MINIMUM → --resolution-forced-minimum when True (not STN lower bounds)
RESOLUTION_ALPHA = 2
RESOLUTION_FORCED_MINIMUM = False

# Runtime benchmark grid — Script 2 (same lists as Script 1 unless you override)
RT_OBJECTS   = OBJECTS
RT_DEADLINES = DEADLINES
RT_MAX_STEPS = 1000
# Per-scenario heuristic depth = deadline (see Script 2 run cell)
RT_SCENARIO_GRID = list(itertools.product(DOMAINS, RT_OBJECTS, RT_DEADLINES))

# Script 1b MCTS tree-inspection milestones: cumulative selection iterations.
MCTS_INSPECT_MILESTONES = [0, 14, 50, 100, 250, 500, 1000,5000]
MCTS_INSPECT_TOP_N      = 12

# Output paths — absolute so download / pandas always match where scripts write
MCTS_CSV      = str((REPO_ROOT / "results" / "mcts_heuristic_comparison.csv").resolve())
RUNTIME_CSV   = str((REPO_ROOT / "results" / "heuristic_runtime_per_call.csv").resolve())
RUNTIME_XLSX  = str((REPO_ROOT / "results" / "heuristic_runtime_per_call.xlsx").resolve())
SCRIPT3_CSV   = str((REPO_ROOT / "results" / "script3_multi_domain_heuristic_comparison.csv").resolve())

print("Config:")
print(f"  REPO_ROOT      : {REPO_ROOT}")
print(f"  MCTS scenarios : domains={DOMAINS}  objects={OBJECTS}  garbage={GARBAGE_AMOUNTS}  deadlines={DEADLINES}  runs={RUNS}  seed={SEED}")
print(f"  MCTS args      : search_time={SEARCH_TIME}  search_depth={SEARCH_DEPTH}  k={K_RANDOM_ACTIONS}  C={EXPLORATION_CONSTANT}  selection={SELECTION_TYPE}")
print(f"  Reward/value   : gamma={DISCOUNT_FACTOR}  step_penalty={STEP_PENALTY}  reward_mode={REWARD_MODE}  value_mode={VALUE_MODE}  final_selection={FINAL_SELECTION}  fixed_tail_h={FIXED_TAIL_H}")
print(f"  Script3 solver : {SCRIPT3_SOLVER}")
print(f"  Heuristics     : {HEURISTICS}")
print(f"  Gamma calib    : {ENABLE_ROLLOUT_CALIBRATION}")
print(f"  Alignment      : dynamic={not ROLLOUT_ALIGNED_FIXED_H}  H(fixed-only)={ROLLOUT_ALIGNED_H}  redo={ROLLOUT_ALIGNED_REDO}  boundary={ROLLOUT_ALIGNED_BOUNDARY}  lambda_align={ROLLOUT_ALIGNED_LAMBDA_ALIGN}")
print(f"  Align budgets  : per_node={ROLLOUT_ALIGNED_MAX_ROLLOUTS_PER_NODE}  per_search={ROLLOUT_ALIGNED_MAX_ROLLOUTS_PER_SEARCH}  time={ROLLOUT_ALIGNED_MAX_TIME_PER_SEARCH}")
print(f"  Option A debug : {FRONTIER_OPTION_A_DEBUG}")
print(f"  Resolution     : alpha={RESOLUTION_ALPHA}  forced_minimum={RESOLUTION_FORCED_MINIMUM}")
print(f"  Tree inspect   : milestones={MCTS_INSPECT_MILESTONES}  top_n={MCTS_INSPECT_TOP_N}")
print(f"  Runtime bench  : domains={DOMAINS}  grid={RT_SCENARIO_GRID}  max_steps={RT_MAX_STEPS}  (depth=deadline per scenario)")
print(f"  Script3 caps   : object_domains={sorted(DOMAINS_USING_OBJECTS)}  garbage_domains={sorted(DOMAINS_USING_GARBAGE)}")
print(f"  MCTS_CSV       : {MCTS_CSV}")
print(f"  RUNTIME_CSV    : {RUNTIME_CSV}")
print(f"  RUNTIME_XLSX   : {RUNTIME_XLSX}")
print(f"  SCRIPT3_CSV    : {SCRIPT3_CSV}")
print(f"  Ptrpg rollout  : max_steps={PTRPG_GUIDED_ROLLOUT_MAX_STEPS}  epsilon={PTRPG_GUIDED_ROLLOUT_EPSILON}  debug={PTRPG_GUIDED_ROLLOUT_DEBUG}")

_require_colab_repo_has_fixed_tail()
validate_heuristics(HEURISTICS)

Config:
  REPO_ROOT      : /content/tp_mcts
  MCTS scenarios : domains=['nasa_rover']  objects=[2]  garbage=[10]  deadlines=[25]  runs=10  seed=123
  MCTS args      : search_time=1  search_depth=10  k=10  C=10  selection=avg
  Reward/value   : gamma=1  step_penalty=0  reward_mode=terminal  value_mode=tp_mcts  final_selection=q  fixed_tail_h=10
  Script3 solver : greedy_parallel
  Heuristics     : ['fixed_tail_atomic_exact_resolution']
  Gamma calib    : False
  Alignment      : dynamic=True  H(fixed-only)=15  redo=1  boundary=wait_no_overshoot  lambda_align=1.0
  Align budgets  : per_node=0  per_search=0  time=0.0
  Option A debug : False
  Resolution     : alpha=2  forced_minimum=False
  Tree inspect   : milestones=[0, 14, 50, 100, 250, 500, 1000, 5000]  top_n=12
  Runtime bench  : domains=['nasa_rover']  grid=[('nasa_rover', 2, 25)]  max_steps=1000  (depth=deadline per scenario)
  Script3 caps   : object_domains=['hosting', 'machine_shop', 'nasa_rover', 'prob_match_cellar', 'stuck_ca

ValueError: Unknown heuristic 'fixed_tail_atomic_exact_resolution'. Valid options: baseline, baseline_cached, baseline_survival, baseline_survival_meanvar, baseline_survival_and_gamma, atomic_exact_resolution_and_gamma, atom_backtrack_exact_resolution_and_gamma, baseline_survival_resolution, rollout_aligned_baseline, rollout_aligned_survival, rollout_aligned_resolution_survival, frontier_aligned_baseline, frontier_aligned_survival, frontier_aligned_resolution_survival, frontier_aligned_option_a, frontier_aligned_option_a_survival, frontier_aligned_option_a_resolution, atomic_exact, atomic_exact_resolution, atom_backtrack_exact_resolution, atomic_exact_unbiased, atom_backtrack_exact_unbiased, atomic_exact_cached, fast_atom_cache, baseline_pessimistic, baseline_passmistic, ptrpg_old, ptrpg_guided_rollout_baseline_survival_resolution, ptrpg_guided_rollout_atomic_exact_resolution

## Script 1 — MCTS Heuristic Comparison

Runs TP-MCTS on each **(object_amount, deadline) × heuristic** combination.

- Results are saved **incrementally** — partial data survives a Colab timeout.
- Output: `results/mcts_heuristic_comparison.csv`

In [ ]:
import os
import subprocess
import pandas as pd

sub_env = os.environ.copy()
sub_env["PYTHONPATH"] = str(REPO_ROOT)

results_dir = REPO_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
part_paths = []
frames = []

script_path = REPO_ROOT / "scripts" / "run_mcts_heuristic_comparison.py"
if not script_path.exists():
    raise FileNotFoundError(f"Could not find script at: {script_path}")

help_proc = subprocess.run(
    ["python", str(script_path), "-h"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=str(REPO_ROOT),
    env=sub_env,
    check=False,
)
supports_k = "--k" in help_proc.stdout
supports_final_selection = "--final_selection" in help_proc.stdout
supports_and_gamma_calib = "--and-gamma-rollout-calibration" in help_proc.stdout
supports_ptrpg_guided = "--ptrpg-guided-rollout-policy" in help_proc.stdout
supports_fixed_tail = "--fixed-tail-h" in help_proc.stdout
# Base alignment knobs (H/redo/budgets) vs the newer dynamic knobs
# (boundary-mode / lambda-align / fixed-h). Gate each independently so a
# partially-updated clone warns instead of crashing.
supports_rollout_aligned = "--rollout-aligned-h" in help_proc.stdout
supports_rollout_aligned_dyn = "--rollout-aligned-boundary-mode" in help_proc.stdout
supports_frontier_option_a_debug = "--frontier-option-a-debug" in help_proc.stdout
_uses_option_a = any(h.startswith("frontier_aligned_option_a") for h in HEURISTICS)
_uses_ptrpg_guided = any(h.startswith("ptrpg_guided_rollout") for h in HEURISTICS) or VALUE_MODE == "ptrpg_guided_terminal_rollout"
_uses_fixed_tail = (
    any(h.startswith("fixed_tail") for h in HEURISTICS)
    or VALUE_MODE == "fixed_tail_ptrpg_rollout"
)

if not supports_k:
    print(
        "[warn] Current run_mcts_heuristic_comparison.py does not support --k; running without it.",
        flush=True,
    )
if not supports_final_selection:
    print(
        "[warn] Current run_mcts_heuristic_comparison.py does not support --final_selection; running without it.",
        flush=True,
    )
if ENABLE_ROLLOUT_CALIBRATION and not supports_and_gamma_calib:
    print(
        "[warn] Current run_mcts_heuristic_comparison.py does not support "
        "--and-gamma-rollout-calibration; running with static gamma instead.",
        flush=True,
    )
_uses_alignment = any(h.startswith(("rollout_aligned", "frontier_aligned")) for h in HEURISTICS)
_AND_GAMMA_HEURISTICS = frozenset({
    "baseline_survival_and_gamma",
    "atomic_exact_resolution_and_gamma",
    "atom_backtrack_exact_resolution_and_gamma",
})
_uses_and_gamma = any(
    h in _AND_GAMMA_HEURISTICS
    or str(HEURISTIC_ALIASES.get(h, {}).get("temporal_heuristic_strategy", "")).endswith("_and_gamma")
    for h in HEURISTICS
)
_needs_resolution = any(
    "resolution" in h
    or str(HEURISTIC_ALIASES.get(h, {}).get("temporal_heuristic_strategy", "")).endswith("_resolution")
    for h in HEURISTICS
)
if _uses_alignment and not supports_rollout_aligned:
    print(
        "[warn] This run_mcts_heuristic_comparison.py has none of the --rollout-aligned-* "
        "knobs; the alignment heuristics will use their defaults. (Push/pull the latest scripts.)",
        flush=True,
    )
elif _uses_alignment and not supports_rollout_aligned_dyn:
    print(
        "[warn] This run_mcts_heuristic_comparison.py is missing the newer alignment knobs "
        "(--rollout-aligned-boundary-mode / --rollout-aligned-lambda-align / --rollout-aligned-fixed-h). "
        "Those are skipped; only H/redo/budgets are applied. (Push/pull the latest scripts to enable them.)",
        flush=True,
    )
if _uses_ptrpg_guided and not supports_ptrpg_guided:
    raise RuntimeError(
        "HEURISTICS or VALUE_MODE request ptrpg_guided_terminal_rollout, but "
        "run_mcts_heuristic_comparison.py is too old (missing --ptrpg-guided-rollout-policy). "
        "Use the local repo scripts/ or git pull."
    )
if _uses_ptrpg_guided and (ENABLE_ROLLOUT_CALIBRATION or _uses_alignment or _uses_option_a):
    print(
        "[warn] ptrpg_guided rollout is combined with alignment and/or and-gamma calibration. "
        "For the first benchmark, set ENABLE_ROLLOUT_CALIBRATION=False and avoid rollout_aligned / "
        "frontier_aligned heuristics in HEURISTICS.",
        flush=True,
    )
if _uses_fixed_tail and not supports_fixed_tail:
    raise RuntimeError(
        "HEURISTICS or VALUE_MODE request fixed_tail_ptrpg_rollout, but "
        "run_mcts_heuristic_comparison.py is too old (missing --fixed-tail-h). "
        "Use the local repo scripts/ or git pull."
    )
if _uses_fixed_tail and (ENABLE_ROLLOUT_CALIBRATION or _uses_alignment or _uses_option_a):
    print(
        "[warn] fixed_tail_ptrpg_rollout is combined with alignment and/or and-gamma calibration. "
        "For the first benchmark, set ENABLE_ROLLOUT_CALIBRATION=False and avoid rollout_aligned / "
        "frontier_aligned heuristics in HEURISTICS.",
        flush=True,
    )

for idx, dom in enumerate(DOMAINS, start=1):
    part_csv = results_dir / f"_mcts_part_{dom}.csv"
    cmd = [
        "python", str(script_path),
        "--runs",                 str(RUNS),
        "--seed",                 str(SEED),
        "--search_time",          str(SEARCH_TIME),
        "--search_depth",         str(SEARCH_DEPTH),
        "--exploration_constant", str(EXPLORATION_CONSTANT),
        "--selection_type",        SELECTION_TYPE,
        "--reward_mode",          REWARD_MODE,
        "--value_mode",           VALUE_MODE,
        "--domain",               dom,
        "--discount_factor",      str(DISCOUNT_FACTOR),
        "--step_penalty",         str(STEP_PENALTY),
        "--output",               str(part_csv.resolve()),
        "--objects",              *[str(o) for o in OBJECTS],
        "--deadlines",            *[str(d) for d in DEADLINES],
        "--heuristics",           *HEURISTICS,
    ]

    if supports_k:
        cmd.extend(["--k", str(K_RANDOM_ACTIONS)])

    if supports_final_selection:
        cmd.extend(["--final_selection", FINAL_SELECTION])

    if _needs_resolution and RESOLUTION_ALPHA is not None:
        cmd.extend(["--resolution-alpha", str(RESOLUTION_ALPHA)])
    if _needs_resolution and RESOLUTION_FORCED_MINIMUM:
        cmd.append("--resolution-forced-minimum")
    if ENABLE_ROLLOUT_CALIBRATION and supports_and_gamma_calib and _uses_and_gamma:
        cmd.append("--and-gamma-rollout-calibration")

    # Base alignment knobs — only when an aligned heuristic is in HEURISTICS.
    if supports_rollout_aligned and _uses_alignment:
        cmd.extend(["--rollout-aligned-h", str(ROLLOUT_ALIGNED_H)])
        cmd.extend(["--rollout-aligned-redo", str(ROLLOUT_ALIGNED_REDO)])
        if ROLLOUT_ALIGNED_MAX_ROLLOUTS_PER_NODE:
            cmd.extend(["--rollout-aligned-max-rollouts-per-node", str(ROLLOUT_ALIGNED_MAX_ROLLOUTS_PER_NODE)])
        if ROLLOUT_ALIGNED_MAX_ROLLOUTS_PER_SEARCH:
            cmd.extend(["--rollout-aligned-max-rollouts-per-search", str(ROLLOUT_ALIGNED_MAX_ROLLOUTS_PER_SEARCH)])
        if ROLLOUT_ALIGNED_MAX_TIME_PER_SEARCH:
            cmd.extend(["--rollout-aligned-max-time-per-search", str(ROLLOUT_ALIGNED_MAX_TIME_PER_SEARCH)])
    # Newer dynamic knobs (boundary / lambda / fixed-h) — aligned heuristics only.
    if supports_rollout_aligned_dyn and _uses_alignment:
        cmd.extend(["--rollout-aligned-boundary-mode", str(ROLLOUT_ALIGNED_BOUNDARY)])
        cmd.extend(["--rollout-aligned-lambda-align", str(ROLLOUT_ALIGNED_LAMBDA_ALIGN)])
        if ROLLOUT_ALIGNED_FIXED_H:
            cmd.append("--rollout-aligned-fixed-h")

    # Rollout policy per heuristic comes from HEURISTIC_ALIASES inside run_mcts_heuristic_comparison.
    if supports_ptrpg_guided and (_uses_ptrpg_guided or VALUE_MODE == "ptrpg_guided_terminal_rollout"):
        if PTRPG_GUIDED_ROLLOUT_MAX_STEPS is not None:
            cmd.extend(["--ptrpg-guided-rollout-max-steps", str(PTRPG_GUIDED_ROLLOUT_MAX_STEPS)])
        if PTRPG_GUIDED_ROLLOUT_EPSILON is not None:
            cmd.extend(["--ptrpg-guided-rollout-epsilon", str(PTRPG_GUIDED_ROLLOUT_EPSILON)])
        if PTRPG_GUIDED_ROLLOUT_DEBUG:
            cmd.append("--ptrpg-guided-rollout-debug")

    if supports_fixed_tail and (_uses_fixed_tail or VALUE_MODE == "fixed_tail_ptrpg_rollout"):
        cmd.extend(["--fixed-tail-h", str(FIXED_TAIL_H)])
        if FIXED_TAIL_DEBUG:
            cmd.append("--fixed-tail-debug")

    if supports_frontier_option_a_debug and FRONTIER_OPTION_A_DEBUG and _uses_option_a:
        cmd.append("--frontier-option-a-debug")

    print(f"\n{'='*60}")
    print(f"  MCTS benchmark  [{idx}/{len(DOMAINS)}]  domain={dom}")
    print(f"{'='*60}")
    print("Using script:", script_path, flush=True)
    print("Running:", " ".join(cmd), flush=True)
    print()

    # Stream output live — cwd=REPO_ROOT so relative paths match this notebook
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(REPO_ROOT),
        env=sub_env,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    print(f"\nProcess exited with code {proc.returncode}")

    if proc.returncode != 0:
        raise RuntimeError(
            f"run_mcts_heuristic_comparison failed for domain={dom} (exit {proc.returncode})"
        )

    part_paths.append(part_csv)
    frames.append(pd.read_csv(part_csv))

if not frames:
    raise RuntimeError("No MCTS outputs were produced; DOMAINS is empty.")

df_mcts_merged = pd.concat(frames, ignore_index=True)
df_mcts_merged.to_csv(MCTS_CSV, index=False)
print(f"\nMerged {len(frames)} domain(s) -> {MCTS_CSV}  ({len(df_mcts_merged)} rows)")

for p in part_paths:
    try:
        p.unlink()
    except OSError:
        pass


  MCTS benchmark  [1/1]  domain=nasa_rover
Using script: /content/tp_mcts/scripts/run_mcts_heuristic_comparison.py
Running: python /content/tp_mcts/scripts/run_mcts_heuristic_comparison.py --runs 10 --seed 123 --search_time 1 --search_depth 40 --exploration_constant 3 --selection_type avg --reward_mode terminal --value_mode tp_mcts --domain nasa_rover --discount_factor 1 --step_penalty 0 --output /content/tp_mcts/results/_mcts_part_nasa_rover.csv --objects 2 --deadlines 25 --heuristics frontier_aligned_option_a_resolution --k 10 --final_selection q --resolution-alpha 2 --rollout-aligned-h 15 --rollout-aligned-redo 1 --rollout-aligned-boundary-mode wait_no_overshoot --rollout-aligned-lambda-align 1.0




  TP-MCTS Heuristic Comparison
  Domain    : nasa_rover
  Scenarios : [(2, 25)]
  Heuristics: ['frontier_aligned_option_a_resolution']
  Per block : --runs 10 (MCTS episodes inside run_domain for ONE row)
  Outer grid: 1 scenarios x 1 heuristics = 1 rows in CSV
  MCTS args : search_time=1  search_depth=40  k=10  C=3.0  selection=avg
  Seed / gamma / reward : seed=123  gamma=1.0  step_penalty=0.0  reward_mode=terminal  value_mode=tp_mcts
  Output    : /content/tp_mcts/results/_mcts_part_nasa_rover.csv

[CSV row 1/1: obj=2 deadline=25 heuristic=frontier_aligned_option_a_resolution | inner loop = 10 MCTS runs]

────────────────────────────────────────────────────────────
  [nasa_rover obj=2 dl=25] heuristic=frontier_aligned_option_a_resolution
  heuristic_depth=25  search_time=1  search_depth=40  k=10  runs=10  seed=123  C=3.0  selection=avg  gamma=1.0  step_penalty=0.0  reward_mode=terminal  value_mode=tp_mcts  final_selection=q
─────────────────────────────────────────────────────────

: 

## Script 1b — Inspect One MCTS Tree

Builds one TP-MCTS root from the current config and grows it by fixed selection-iteration milestones. Use this to see whether the returned action is actually expanded or only heuristic-initialized by `selection_type="max"`.

**Run order:** run **Config** above before this cell (defines `MCTS_INSPECT_MILESTONES` / `REPO_ROOT`). **Ignore gray saved output** in the notebook from older runs — only a fresh run’s `Running:` line matches your current source. On Colab, `REPO_ROOT` should be `/content/tp_mcts` (the clone root); a nested `tp_mcts/tp_mcts` copy is not used when the outer tree has `experiments.ipynb`.

In [11]:
import subprocess
import sys

inspect_script = REPO_ROOT / "scripts" / "inspect_mcts_tree.py"
if not inspect_script.exists():
    raise FileNotFoundError(f"Could not find script at: {inspect_script}")

debug_object = OBJECTS[0]
debug_deadline = DEADLINES[0]
debug_heuristic = HEURISTICS[0]

# Pick up latest scripts/experiment_common.py (e.g. after git pull without re-running Config).
import importlib
import experiment_common as _experiment_common
importlib.reload(_experiment_common)
from experiment_common import HEURISTIC_ALIASES, validate_heuristics

validate_heuristics([debug_heuristic])
_debug_alias = HEURISTIC_ALIASES[debug_heuristic]
_inspect_value_mode = _debug_alias.get("value_mode", VALUE_MODE)

cmd = [
    sys.executable, str(inspect_script),
    "--domain",               DOMAIN,
    "--object_amount",        str(debug_object),
    "--deadline",             str(debug_deadline),
    "--heuristic",            debug_heuristic,
    "--selection_type",       SELECTION_TYPE,
    "--value_mode",           _inspect_value_mode,
    "--search_depth",         str(SEARCH_DEPTH),
    "--k",                    str(K_RANDOM_ACTIONS),
    "--exploration_constant", str(EXPLORATION_CONSTANT),
    "--discount_factor",      str(DISCOUNT_FACTOR),
    "--step_penalty",         str(STEP_PENALTY),
    "--reward_mode",          REWARD_MODE,
    "--seed",                 str(SEED),
    "--milestones",           *[str(m) for m in MCTS_INSPECT_MILESTONES],
    "--top_n",                str(MCTS_INSPECT_TOP_N),
    "--expected",
]

if RESOLUTION_ALPHA is not None:
    cmd.extend(["--resolution-alpha", str(RESOLUTION_ALPHA)])
if RESOLUTION_FORCED_MINIMUM:
    cmd.append("--resolution-forced-minimum")

_inspect_uses_alignment = debug_heuristic.startswith(("rollout_aligned", "frontier_aligned"))
if _inspect_uses_alignment:
    cmd.extend(["--rollout-aligned-h", str(ROLLOUT_ALIGNED_H)])
    cmd.extend(["--rollout-aligned-redo", str(ROLLOUT_ALIGNED_REDO)])
    cmd.extend(["--rollout-aligned-boundary-mode", str(ROLLOUT_ALIGNED_BOUNDARY)])
    cmd.extend(["--rollout-aligned-lambda-align", str(ROLLOUT_ALIGNED_LAMBDA_ALIGN)])
    if ROLLOUT_ALIGNED_FIXED_H:
        cmd.append("--rollout-aligned-fixed-h")

_inspect_uses_ptrpg = (
    debug_heuristic.startswith("ptrpg_guided_rollout")
    or _inspect_value_mode == "ptrpg_guided_terminal_rollout"
)
_inspect_uses_fixed_tail = (
    debug_heuristic.startswith("fixed_tail")
    or _inspect_value_mode == "fixed_tail_ptrpg_rollout"
)
if _inspect_uses_ptrpg or _inspect_uses_fixed_tail:
    _policy = _debug_alias.get("ptrpg_guided_rollout_policy")
    if _policy is not None:
        cmd.extend(["--ptrpg-guided-rollout-policy", str(_policy)])
if _inspect_uses_fixed_tail:
    cmd.extend(["--fixed-tail-h", str(_debug_alias.get("fixed_tail_h", FIXED_TAIL_H))])
    if FIXED_TAIL_DEBUG:
        cmd.append("--fixed-tail-debug")

print("REPO_ROOT:", REPO_ROOT, flush=True)
print("Inspector:", inspect_script, flush=True)
print("Milestones:", MCTS_INSPECT_MILESTONES, flush=True)
git_proc = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "log", "-1", "--oneline"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)
print("Git HEAD:", git_proc.stdout.strip(), flush=True)
print("Running:", " ".join(cmd), flush=True)
print()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=str(REPO_ROOT),
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nProcess exited with code {proc.returncode}")

KeyError: 'fixed_tail_atomic_exact_resolution'

In [14]:
import pandas as pd

df = pd.read_csv(MCTS_CSV)

# Pivot: rows = (domain, objects, deadline), columns = heuristic, values = success_rate
pivot = df.pivot_table(
    index=["domain", "object_amount", "deadline"],
    columns="heuristic",
    values="success_rate",
    aggfunc="first",
)
print("=== Success rate by scenario × heuristic ===")
print(pivot.to_string())

print("\n=== Full results table ===")
cols = ["domain", "object_amount", "deadline", "heuristic", "selection_type", "discount_factor", "step_penalty",
        "amount_success", "success_rate", "avg_success_time", "std_success_time"]
cols = [c for c in cols if c in df.columns]
print(df[cols].to_string(index=False))

=== Success rate by scenario × heuristic ===
heuristic               atomic_exact
object_amount deadline              
2             25                0.65

=== Full results table ===
    domain  object_amount  deadline    heuristic selection_type  discount_factor  step_penalty  amount_success  success_rate  avg_success_time  std_success_time
nasa_rover              2        25 atomic_exact            max              1.0           0.0              13          0.65         17.461538          0.461538


## Script 2 — Heuristic Per-Call Runtime Benchmark

For each **(object_amount, deadline)** in the configured grid, runs `greedy_parallel` for each heuristic and measures:

- `wrapper_avg_call_sec` — total heuristic call cost (includes STN work)
- `worker_avg_call_sec` — pure propagation cost
- `worker_cache_hit_avg_sec` / `worker_cache_miss_avg_sec` — cache breakdown

Outputs (merged across scenarios): `results/heuristic_runtime_per_call.csv`, and `results/heuristic_runtime_per_call.xlsx` with blank rows between scenarios (written in the results cell below).

In [3]:
import subprocess
import pandas as pd

results_dir = REPO_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
part_paths = []
frames = []

for idx, (dom, obj, dl) in enumerate(RT_SCENARIO_GRID, start=1):
    part_csv = results_dir / f"_runtime_part_{dom}_{obj}_{dl}.csv"
    cmd = [
        "python", "scripts/run_heuristic_runtime_per_call.py",
        "--domain",         dom,
        "--object_amount",  str(obj),
        "--deadline",       str(dl),
        "--heuristic_depth",str(dl),
        "--max_steps",      str(RT_MAX_STEPS),
        "--seed",           str(SEED),
        "--reward_mode",    REWARD_MODE,
        "--discount_factor",str(DISCOUNT_FACTOR),
        "--step_penalty",   str(STEP_PENALTY),
        "--output",         str(part_csv.resolve()),
        "--heuristics",     *HEURISTICS,
    ]
    if RESOLUTION_ALPHA is not None:
        cmd.extend(["--resolution-alpha", str(RESOLUTION_ALPHA)])
    if RESOLUTION_FORCED_MINIMUM:
        cmd.append("--resolution-forced-minimum")

    print(f"\n{'='*60}")
    print(f"  Runtime benchmark  [{idx}/{len(RT_SCENARIO_GRID)}]  domain={dom}  obj={obj}  deadline={dl}  depth={dl}")
    print(f"{'='*60}")
    print("Running:", " ".join(cmd), flush=True)
    print()

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(REPO_ROOT),
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    print(f"\nProcess exited with code {proc.returncode}")

    if proc.returncode != 0:
        raise RuntimeError(
            f"run_heuristic_runtime_per_call failed for domain={dom} obj={obj} deadline={dl} (exit {proc.returncode})"
        )

    part_paths.append(part_csv)
    frames.append(pd.read_csv(part_csv))

df_rt_merged = pd.concat(frames, ignore_index=True)
df_rt_merged.to_csv(RUNTIME_CSV, index=False)
print(f"\nMerged {len(frames)} scenario(s) -> {RUNTIME_CSV}  ({len(df_rt_merged)} rows)")

for p in part_paths:
    try:
        p.unlink()
    except OSError:
        pass


  Runtime benchmark  [1/1]  domain=nasa_rover  obj=2  deadline=25  depth=25
Running: python scripts/run_heuristic_runtime_per_call.py --domain nasa_rover --object_amount 2 --deadline 25 --heuristic_depth 25 --max_steps 1000 --seed 123 --reward_mode terminal --discount_factor 1 --step_penalty 0 --output /content/tp_mcts/results/_runtime_part_nasa_rover_2_25.csv --heuristics atomic_exact_unbiased --resolution-alpha 2


  Heuristic Per-Call Runtime Benchmark
  Scenario  : nasa_rover  obj=2  deadline=25
  H-depth   : 25  max_steps=1000  seed=123  reward=terminal  gamma=1.0  step_penalty=0.0
  Heuristics: ['atomic_exact_unbiased']
  Output    : /content/tp_mcts/results/_runtime_part_nasa_rover_2_25.csv

  Heuristic : atomic_exact_unbiased  (atomic_exact_unbiased)
  Internal  : heuristic_name=temporal_probabilistic_rpg  strategy=atom_backtrack_exact_unbiased
  Scenario  : nasa_rover obj=2  deadline=25  depth=25
started step 0
Current state is state: store_of(s0, r0) ; store_of(s1, r0) ; on_

In [4]:
import pandas as pd

df_rt = pd.read_csv(RUNTIME_CSV)

scenario_cols = ["domain", "object_amount", "deadline"]
timing_cols = [
    "heuristic",
    "wrapper_avg_call_sec",
    "worker_avg_call_sec",
    "worker_cache_hit_avg_sec",
    "worker_cache_miss_avg_sec",
    "worker_cache_hits",
    "worker_cache_misses",
    "plan_success",
]
out_cols = scenario_cols + timing_cols

spaced_parts = []
for i, (dom, obj, dl) in enumerate(RT_SCENARIO_GRID):
    mask = (
        (df_rt["domain"] == dom)
        & (df_rt["object_amount"] == obj)
        & (df_rt["deadline"] == dl)
    )
    df_grp = df_rt.loc[mask, timing_cols].sort_values("wrapper_avg_call_sec")
    df_grp.insert(0, "deadline", dl)
    df_grp.insert(0, "object_amount", obj)
    df_grp.insert(0, "domain", dom)
    print(f"=== domain={dom}  obj={obj}  deadline={dl}  (fastest → slowest) ===")
    print(df_grp.to_string(index=False))
    print()
    spaced_parts.append(df_grp)
    if i < len(RT_SCENARIO_GRID) - 1:
        spaced_parts.append(pd.DataFrame([{c: float("nan") for c in out_cols}]))

df_runtime_xlsx = pd.concat(spaced_parts, ignore_index=True)
with pd.ExcelWriter(RUNTIME_XLSX, engine="openpyxl") as writer:
    df_runtime_xlsx.to_excel(writer, sheet_name="runtime_ranking", index=False)
print(f"Wrote Excel (blank rows between scenarios): {RUNTIME_XLSX}")

=== domain=nasa_rover  obj=2  deadline=25  (fastest → slowest) ===
    domain  object_amount  deadline             heuristic  wrapper_avg_call_sec  worker_avg_call_sec  worker_cache_hit_avg_sec  worker_cache_miss_avg_sec  worker_cache_hits  worker_cache_misses  plan_success
nasa_rover              2        25 atomic_exact_unbiased              0.058036              0.05773                  0.000045                   0.063347                 52                  534          True

Wrote Excel (blank rows between scenarios): /content/tp_mcts/results/heuristic_runtime_per_call.xlsx


## Download Results to your PC

Run this cell after any experiment.

- **On Colab**: Colab cannot write to a path on your PC. This cell triggers a **browser download** (usually to **Downloads**). To land files in `TP_MCTS\results` on Windows, either move them after download, or set your browser’s default download folder to that directory (Chrome: Settings → Downloads → Location).
- **Local (this repo on your machine)**: copies each CSV into `LOCAL_PC_RESULTS_DIR` (default: `C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results`). Override with env var `TP_MCTS_RESULTS_DIR`.

Re-run the **Config** cell first so `MCTS_CSV`, `RUNTIME_CSV`, and `RUNTIME_XLSX` are absolute paths under `REPO_ROOT`. If Script 1 was never run in this session, the MCTS CSV is skipped until you run it.

In [5]:
from pathlib import Path
import os
import shutil


def _find_repo_root_dl() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "run_mcts_heuristic_comparison.py").is_file():
            return p
    return start


# Local copy destination (not Colab). Override with env TP_MCTS_RESULTS_DIR.
if "TP_MCTS_RESULTS_DIR" in os.environ:
    LOCAL_PC_RESULTS_DIR = Path(os.environ["TP_MCTS_RESULTS_DIR"])
elif os.name == "nt":
    LOCAL_PC_RESULTS_DIR = Path(r"C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results")
else:
    LOCAL_PC_RESULTS_DIR = _find_repo_root_dl() / "results"
# Hint on Colab (Linux VM): where to put files on your PC; override with TP_MCTS_RESULTS_DIR.
_PC_RESULTS_HINT = (
    Path(os.environ["TP_MCTS_RESULTS_DIR"])
    if "TP_MCTS_RESULTS_DIR" in os.environ
    else Path(r"C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results")
)


def _resolve_csv(path: str) -> Path | None:
    """Resolve CSV path even if kernel cwd != repo root (Colab / multi-root)."""
    p = Path(path)
    if p.is_file():
        return p
    alt = _find_repo_root_dl() / "results" / p.name
    if alt.is_file():
        return alt
    return None


def _is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def download_result(path: str) -> None:
    """Colab: browser download. Local: copy into LOCAL_PC_RESULTS_DIR."""
    p = _resolve_csv(path)
    if p is None:
        print(f"  [skip] not found: {path}")
        print(f"          tried: {Path(path).resolve()} and {_find_repo_root_dl() / 'results' / Path(path).name}")
        return
    sp = str(p.resolve())
    size_kb = p.stat().st_size / 1024

    if _is_colab():
        from google.colab import files

        files.download(sp)
        print(f"  [browser download] {sp}  ({size_kb:.1f} KB)")
        print(f"      → On your PC, move the file to: {_PC_RESULTS_HINT}")
        print("      (Or set Chrome/Edge default download folder to that path.)")
        return

    LOCAL_PC_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    dest = LOCAL_PC_RESULTS_DIR / p.name
    shutil.copy2(p, dest)
    print(f"  [copied] {sp}  ({size_kb:.1f} KB)")
    print(f"      → {dest.resolve()}")


print("Exporting results...")
download_result(MCTS_CSV)
download_result(RUNTIME_CSV)
download_result(RUNTIME_XLSX)
print("Done.")

Exporting results...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  [browser download] /content/tp_mcts/results/mcts_heuristic_comparison.csv  (0.4 KB)
      → On your PC, move the file to: C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results
      (Or set Chrome/Edge default download folder to that path.)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  [browser download] /content/tp_mcts/results/heuristic_runtime_per_call.csv  (0.7 KB)
      → On your PC, move the file to: C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results
      (Or set Chrome/Edge default download folder to that path.)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  [browser download] /content/tp_mcts/results/heuristic_runtime_per_call.xlsx  (5.0 KB)
      → On your PC, move the file to: C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results
      (Or set Chrome/Edge default download folder to that path.)
Done.


### If you cannot find results (Colab)

- **`/content` is erased** when the runtime restarts or disconnects. If you open Colab later without re-running the experiment cells, **`tp_mcts/results` will be empty** — there is nothing to download.
- **Downloads**: the file may be blocked, or saved under another browser profile / “Ask where to save” folder.

**Run the next cell** in the same session **after** experiments: it checks whether the CSVs exist, lists `results/`, and **shows the tables inside the notebook** (works even when download fails).

**To keep files across sessions**, mount Google Drive and copy `results/` there (see comments in that cell), or download from the **Files** sidebar: `content` → `tp_mcts` → `results` → right‑click → **Download**.

In [6]:
# Locate results + show CSVs in the notebook (works on Colab without using Downloads).
from pathlib import Path

try:
    _rr = REPO_ROOT
except NameError:
    raise RuntimeError(
        "Run the **Config** cell first (defines REPO_ROOT, MCTS_CSV, RUNTIME_CSV, RUNTIME_XLSX)."
    ) from None

results_dir = Path(REPO_ROOT) / "results"
print(f"REPO_ROOT     : {REPO_ROOT}")
print(f"results folder: {results_dir}  (exists={results_dir.is_dir()})")
print()

for label, path_str in [
    ("MCTS_CSV", MCTS_CSV),
    ("RUNTIME_CSV", RUNTIME_CSV),
    ("RUNTIME_XLSX", RUNTIME_XLSX),
]:
    p = Path(path_str)
    st = "OK " if p.is_file() else "MISSING — run the experiment script cell in this session"
    print(f"{st}  {label}: {p}")

if results_dir.is_dir():
    print("\nFiles in results/:")
    for f in sorted(results_dir.iterdir()):
        if f.is_file():
            print(f"  {f.name}  ({f.stat().st_size} bytes)")

import pandas as pd
from IPython.display import display

for label, path_str in [("MCTS comparison", MCTS_CSV), ("Heuristic runtime", RUNTIME_CSV)]:
    p = Path(path_str)
    if p.is_file():
        df = pd.read_csv(p)
        print(f"\n=== {label}: {p.name} ({len(df)} rows) ===")
        display(df)
    else:
        print(f"\n=== {label}: skip (file not found) ===")

# --- Optional: persist on Google Drive (uncomment, run mount cell first) ---
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# dest = Path("/content/drive/MyDrive/Colab_TP_MCTS_results")
# dest.mkdir(parents=True, exist_ok=True)
# for path_str in [MCTS_CSV, RUNTIME_CSV, RUNTIME_XLSX]:
#     p = Path(path_str)
#     if p.is_file():
#         shutil.copy2(p, dest / p.name)
#         print("Copied to Drive:", dest / p.name)

REPO_ROOT     : /content/tp_mcts
results folder: /content/tp_mcts/results  (exists=True)

OK   MCTS_CSV: /content/tp_mcts/results/mcts_heuristic_comparison.csv
OK   RUNTIME_CSV: /content/tp_mcts/results/heuristic_runtime_per_call.csv
OK   RUNTIME_XLSX: /content/tp_mcts/results/heuristic_runtime_per_call.xlsx

Files in results/:
  _runtime_part_2_25.csv  (1507 bytes)
  _runtime_part_2_35.csv  (1513 bytes)
  _runtime_part_3_25.csv  (1549 bytes)
  _runtime_part_3_35.csv  (712 bytes)
  diagnostic_mcts.csv  (352 bytes)
  heuristic_runtime_per_call.csv  (697 bytes)
  heuristic_runtime_per_call.xlsx  (5159 bytes)
  mcts_heuristic_comparison.csv  (389 bytes)

=== MCTS comparison: mcts_heuristic_comparison.csv (1 rows) ===


,domain,object_amount,deadline,heuristic,heuristic_label,runs_total,amount_success,success_rate,avg_success_time,std_success_time,...,search_depth,k,selection_type,exploration_constant,discount_factor,step_penalty,reward_mode,value_mode,heuristic_depth,returncode
0,nasa_rover,2,25,atomic_exact,atomic_exact,10,2,0.2,23,1.0,...,3,3,avg,0.0,1.0,0.0,terminal,greedy_matched,25,0



=== Heuristic runtime: heuristic_runtime_per_call.csv (1 rows) ===


,heuristic,heuristic_label,heuristic_name_internal,strategy_internal,domain,object_amount,deadline,reward_mode,discount_factor,step_penalty,...,wrapper_first_call_sec,wrapper_avg_call_sec,worker_total_calls,worker_total_time_sec,worker_first_call_sec,worker_avg_call_sec,worker_cache_hits,worker_cache_misses,worker_cache_hit_avg_sec,worker_cache_miss_avg_sec
0,atomic_exact_unbiased,atomic_exact_unbiased,temporal_probabilistic_rpg,atom_backtrack_exact_unbiased,nasa_rover,2,25,terminal,1.0,0.0,...,0.357612,0.058036,586,33.829492,0.191788,0.05773,52,534,0.000045,0.063347


## Script 3 — Multi-domain Heuristic Sweep (domain-aware args)

Runs multiple heuristics across multiple domains in one run, while varying only the arguments each domain actually uses:
- object-sensitive domains sweep `OBJECTS`
- garbage-sensitive domains sweep `GARBAGE_AMOUNTS`
- domains using both sweep both
- domains using neither run once per deadline

In [ ]:
# Script 3 runner: domain-aware heuristic sweep with garbage support.
import sys
import time
from pathlib import Path
import pandas as pd

scripts_dir = str((Path(REPO_ROOT) / "scripts").resolve())
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

from experiment_common import (
    HEURISTIC_ALIASES,
    validate_heuristics,
    run_domain_subprocess,
    parse_run_metrics,
)

validate_heuristics(HEURISTICS)

SCRIPT3_SMOKE_ONLY = False  # True => quick validation: 1 run, first heuristic, first deadline


def _domain_arg_grid(domain: str):
    uses_obj = domain in DOMAINS_USING_OBJECTS
    uses_garbage = domain in DOMAINS_USING_GARBAGE

    obj_values = OBJECTS if uses_obj else [1]
    garbage_values = GARBAGE_AMOUNTS if uses_garbage else [0]

    for obj in obj_values:
        for garb in garbage_values:
            for dl in DEADLINES:
                yield {
                    "domain": domain,
                    "object_amount": int(obj),
                    "garbage_amount": int(garb),
                    "deadline": int(dl),
                    "uses_object": uses_obj,
                    "uses_garbage": uses_garbage,
                }


scenario_rows = []
for dom in DOMAINS:
    scenario_rows.extend(list(_domain_arg_grid(dom)))

print(f"Script 3 scenarios: {len(scenario_rows)}")
print(pd.DataFrame(scenario_rows).sort_values(["domain", "object_amount", "garbage_amount", "deadline"]).to_string(index=False))

heuristics_to_run = HEURISTICS
runs_to_use = RUNS
if SCRIPT3_SMOKE_ONLY:
    heuristics_to_run = HEURISTICS[:1]
    runs_to_use = 1
    print("[smoke] Running reduced Script 3: first heuristic, RUNS=1")

results = []
total = len(scenario_rows) * len(heuristics_to_run)
completed = 0

for sc in scenario_rows:
    for h in heuristics_to_run:
        completed += 1
        alias = HEURISTIC_ALIASES[h]
        depth = int(sc["deadline"])

        print("\n" + "=" * 76)
        print(
            f"[Script3 {completed}/{total}] solver={SCRIPT3_SOLVER} domain={sc['domain']} obj={sc['object_amount']} "
            f"garbage={sc['garbage_amount']} deadline={sc['deadline']} heuristic={h}"
        )
        print("=" * 76)

        t0 = time.perf_counter()
        output, returncode = run_domain_subprocess(
            domain=sc["domain"],
            object_amount=sc["object_amount"],
            garbage_amount=sc["garbage_amount"],
            deadline=sc["deadline"],
            runs=runs_to_use,
            seed=SEED,
            solver=SCRIPT3_SOLVER,
            heuristic_name=alias["heuristic_name"],
            temporal_heuristic_strategy=alias["temporal_heuristic_strategy"],
            temporal_heuristic_depth=depth,
            search_time=SEARCH_TIME,
            search_depth=SEARCH_DEPTH,
            k=K_RANDOM_ACTIONS,
            selection_type=SELECTION_TYPE,
            exploration_constant=EXPLORATION_CONSTANT,
            reward_mode=REWARD_MODE,
            discount_factor=DISCOUNT_FACTOR,
            step_penalty=STEP_PENALTY,
            value_mode=alias.get("value_mode", VALUE_MODE),
            final_selection=FINAL_SELECTION,
            ptrpg_guided_rollout_policy=alias.get("ptrpg_guided_rollout_policy"),
            fixed_tail_h=alias.get("fixed_tail_h", FIXED_TAIL_H),
            fixed_tail_debug=FIXED_TAIL_DEBUG,
            resolution_alpha=RESOLUTION_ALPHA,
            resolution_forced_minimum=RESOLUTION_FORCED_MINIMUM,
            resolution_reference_t=None,
            verbose=False,
        )
        elapsed = time.perf_counter() - t0
        metrics = parse_run_metrics(output)

        if returncode != 0:
            print(f"[WARN] returncode={returncode}")
            tail_lines = [ln for ln in output.splitlines() if ln.strip()][-8:]
            for ln in tail_lines:
                print("  " + ln)

        row = {
            "solver": SCRIPT3_SOLVER,
            "domain": sc["domain"],
            "object_amount": sc["object_amount"],
            "garbage_amount": sc["garbage_amount"],
            "deadline": sc["deadline"],
            "heuristic": h,
            "heuristic_label": alias["label"],
            "heuristic_name_internal": alias["heuristic_name"],
            "strategy_internal": alias["temporal_heuristic_strategy"],
            "runs_total": metrics.get("runs_total"),
            "amount_success": metrics.get("amount_success"),
            "success_rate": metrics.get("success_rate"),
            "avg_success_time": metrics.get("avg_success_time"),
            "std_success_time": metrics.get("std_success_time"),
            "seed": SEED,
            "search_time": SEARCH_TIME,
            "search_depth": SEARCH_DEPTH,
            "k": K_RANDOM_ACTIONS,
            "selection_type": SELECTION_TYPE,
            "exploration_constant": EXPLORATION_CONSTANT,
            "discount_factor": DISCOUNT_FACTOR,
            "step_penalty": STEP_PENALTY,
            "reward_mode": REWARD_MODE,
            "value_mode": VALUE_MODE,
            "final_selection": FINAL_SELECTION,
            "heuristic_depth": depth,
            "resolution_alpha": RESOLUTION_ALPHA,
            "resolution_forced_minimum": RESOLUTION_FORCED_MINIMUM,
            "returncode": returncode,
            "wall_time_sec": round(elapsed, 3),
        }
        results.append(row)

        pd.DataFrame(results).to_csv(SCRIPT3_CSV, index=False)
        print(
            f"=> success={row['amount_success']}/{row['runs_total']} "
            f"rate={row['success_rate']} avg_time={row['avg_success_time']} wall={row['wall_time_sec']}s"
        )

print(f"\nDone. Script 3 CSV saved to: {SCRIPT3_CSV}")

In [ ]:
# Script 3 results: full table + pivot score views.
from pathlib import Path
import pandas as pd
from IPython.display import display

csv_path = Path(SCRIPT3_CSV)
if not csv_path.is_file():
    raise FileNotFoundError(f"Script 3 CSV not found: {csv_path}. Run Script 3 cell first.")

df = pd.read_csv(csv_path)

common_cols = [
    "solver", "heuristic", "domain", "object_amount", "garbage_amount", "deadline",
    "runs_total", "amount_success", "success_rate", "avg_success_time", "std_success_time",
    "reward_mode", "discount_factor", "step_penalty", "selection_type", "k", "search_time", "search_depth",
]
existing_common_cols = [c for c in common_cols if c in df.columns]

df_sorted = df.sort_values(["heuristic", "domain", "object_amount", "garbage_amount", "deadline"]).reset_index(drop=True)
print(f"Script 3 rows: {len(df_sorted)}")
print(f"Source: {csv_path}")
print("\n=== Full results table (all-common metrics) ===")
display(df_sorted[existing_common_cols])

scenario_cols = ["domain", "object_amount", "garbage_amount", "deadline"]
df_sorted["scenario_key"] = df_sorted[scenario_cols].astype(str).agg(" | ".join, axis=1)

print("\n=== Pivot: success_rate by heuristic × scenario ===")
pivot_success = df_sorted.pivot_table(index="heuristic", columns="scenario_key", values="success_rate", aggfunc="first")
display(pivot_success)

print("\n=== Pivot: avg_success_time by heuristic × scenario ===")
pivot_time = df_sorted.pivot_table(index="heuristic", columns="scenario_key", values="avg_success_time", aggfunc="first")
display(pivot_time)

Script 3 rows: 3
Source: /content/tp_mcts/results/script3_multi_domain_heuristic_comparison.csv

=== Full results table (all-common metrics) ===


,solver,heuristic,domain,object_amount,garbage_amount,deadline,runs_total,amount_success,success_rate,avg_success_time,std_success_time,reward_mode,discount_factor,step_penalty,selection_type,k,search_time,search_depth
0,greedy_parallel,atomic_exact_unbiased,conc,1,0,25,10,10,1.0,15.000000,0.000000,terminal,1,0,avg,10,1,40
1,greedy_parallel,atomic_exact_unbiased,nasa_rover,2,0,25,10,7,0.7,17.857143,0.986301,terminal,1,0,avg,10,1,40
2,greedy_parallel,atomic_exact_unbiased,nasa_rover,3,0,25,10,7,0.7,22.285714,0.865043,terminal,1,0,avg,10,1,40



=== Pivot: success_rate by heuristic × scenario ===


scenario_key,conc | 1 | 0 | 25,nasa_rover | 2 | 0 | 25,nasa_rover | 3 | 0 | 25
heuristic,,,
atomic_exact_unbiased,1.0,0.7,0.7



=== Pivot: avg_success_time by heuristic × scenario ===


scenario_key,conc | 1 | 0 | 25,nasa_rover | 2 | 0 | 25,nasa_rover | 3 | 0 | 25
heuristic,,,
atomic_exact_unbiased,15.0,17.857143,22.285714
